# 07 — Entraînement & Optimisation ML

### Objectif

Construire un pipeline ML **propre**, sans fuite de données, avec une séparation temporelle correcte.

Ce notebook couvre :

- **Étape 4** : modèles de référence (baseline) — comparaison honnête, sans optimisation d'hyperparamètres.
- **Étape 5** : optimisation des hyperparamètres avec validation temporelle.

> Les étapes 1 à 3 (chargement, nettoyage, feature engineering) sont dans les notebooks précédents. On repart du dataset `orbit_dataset_engineering.parquet`.

In [1]:
import time
import warnings
import os

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Bibliothèques chargées.")

Bibliothèques chargées.


#### Chargement du dataset

On charge le dataset d'ingénierie (déjà trié par date).

In [2]:
TARGET = "fillRate_target_t_plus_1"
DATE_SPLIT = "2025-03-15"

data = pd.read_parquet(
    "../data/processed/orbit_dataset_engineering.parquet"
)
data["date_collecte"] = pd.to_datetime(data["date_collecte"])
data = data.sort_values("date_collecte").reset_index(drop=True)

print("Dataset chargé :", data.shape)

Dataset chargé : (116720, 51)


### 4.1 — Corriger la séparation temporelle

Le problème identifié lors des étapes précédentes : la séparation au 15/03/2025 mélangeait des lignes temporelles. On corrige ici.

Résultat attendu :

```
TRAIN : 2022-01-03 → 2025-03-14
TEST  : 2025-03-15 → 2025-12-31
```

In [3]:
# Frontière temporelle
train_data = data[data["date_collecte"] < DATE_SPLIT].copy()
test_data = data[data["date_collecte"] >= DATE_SPLIT].copy()

print("========== NOUVELLE SÉPARATION ==========")
print("TRAIN :", train_data.shape)
print("TEST  :", test_data.shape)

print("\nTRAIN :")
print(train_data["date_collecte"].min(), "->", train_data["date_collecte"].max())

print("\nTEST :")
print(test_data["date_collecte"].min(), "->", test_data["date_collecte"].max())

========== NOUVELLE SÉPARATION ==========
TRAIN : (93360, 51)
TEST  : (23360, 51)

TRAIN :
2022-01-03 00:00:00 -> 2025-03-14 00:00:00

TEST :
2025-03-15 00:00:00 -> 2025-12-31 00:00:00


### 4.1-bis — Correction d'une fuite de données

Deux colonnes du dataset brut encodent quasi directement la cible `fillRate_target_t_plus_1` :

| Colonne | Corrélation cible | Corrélation fillRate courant |
|---|---|---|
| `statut_prevu` | **0.8495** | 0.5451 |
| `priorite_recommandee` | **0.8330** | 0.5150 |

Ce sont **deux encodages de la même variable** (mêmes effectifs par niveau) dont la cible moyenne croît avec le niveau (16.8 → 96.4). Un modèle qui les utilise « prédit » la cible à travers ces colonnes : c'est une **fuite de données** (le statut/la priorité ne sont connus qu'après le remplissage).

**Décision :** on les exclut du pipeline ML. Le R² sera plus bas mais honnête et généralisable.

In [4]:
leak = data[["statut_prevu", "priorite_recommandee", "fillRate", TARGET]].corr()

print("Corrélation des colonnes suspectes :")
print(leak[["fillRate", TARGET]].round(4))

Corrélation des colonnes suspectes :
                          fillRate  fillRate_target_t_plus_1
statut_prevu                0.5451                    0.8495
priorite_recommandee        0.5150                    0.8330
fillRate                    1.0000                    0.4487
fillRate_target_t_plus_1    0.4487                    1.0000


### 4.2 — Construire X et y

In [5]:
# Features = toutes les colonnes sauf identifiants / date / cible / colonnes fuyantes
COLONNES_FUITE = ["statut_prevu", "priorite_recommandee"]

FEATURES = [
    c for c in data.columns
    if c not in ["id_point", "date_collecte", TARGET] + COLONNES_FUITE
]

print("Features utilisées (après correction de la fuite) :", len(FEATURES))
print(sorted(FEATURES))

X_train = train_data[FEATURES].copy()
y_train = train_data[TARGET].copy()

X_test = test_data[FEATURES].copy()
y_test = test_data[TARGET].copy()

print("X_train :", X_train.shape)
print("y_train :", y_train.shape)

print("X_test  :", X_test.shape)
print("y_test  :", y_test.shape)

Features utilisées (après correction de la fuite) : 46
['arrondissement_1', 'arrondissement_2', 'arrondissement_3', 'arrondissement_4', 'capacity_m3', 'charge_precollecteur', 'delta_fillRate_24h', 'densite_population_hab_km2', 'fillRate', 'fillRate_t_minus_1', 'fillRate_t_minus_2', 'humidite_pct', 'indice_meteo', 'indice_priorite', 'indice_saturation', 'is_jour_marche', 'is_weekend', 'jour_semaine_num', 'jours_depuis_derniere_collecte', 'latitude', 'longitude', 'nb_plaintes', 'nb_precollecteurs_dispo', 'nb_signalements_citoyens', 'poids_kg', 'pression_citoyenne', 'pression_collecte', 'quartier_1', 'quartier_10', 'quartier_11', 'quartier_12', 'quartier_13', 'quartier_14', 'quartier_2', 'quartier_3', 'quartier_4', 'quartier_5', 'quartier_6', 'quartier_7', 'quartier_8', 'quartier_9', 'ratio_plaintes', 'rendement_remplissage', 'risque_debordement', 'rolling_mean_3d', 'temperature_celsius']
X_train : (93360, 46)
y_train : (93360,)
X_test  : (23360, 46)
y_test  : (23360,)


In [6]:
# Les dummies booléennes deviennent des entiers (0/1) pour homogénéité
for df_xy in (X_train, X_test):
    bool_cols = df_xy.select_dtypes(include=["bool"]).columns
    df_xy[bool_cols] = df_xy[bool_cols].astype(int)

### 4.3 — Vérifier les valeurs manquantes

In [7]:
print("========== VALEURS MANQUANTES ==========")
print("Train :", X_train.isna().sum().sum())
print("Test  :", X_test.isna().sum().sum())

print("\nCible train :", y_train.isna().sum())
print("Cible test  :", y_test.isna().sum())

========== VALEURS MANQUANTES ==========
Train : 0
Test  : 0

Cible train : 0
Cible test  : 0


### 4.4 — Vérifier les variables catégorielles

Résultat attendu : `[]` (tout est numérique/dummies).

In [8]:
print("========== TYPES DES FEATURES ==========")
print(X_train.dtypes.value_counts())

print("\nVariables non numériques :")
print(X_train.select_dtypes(exclude=["number"]).columns.tolist())

========== TYPES DES FEATURES ==========
int64      28
float64    18
Name: count, dtype: int64

Variables non numériques :
[]


### 4.5 — Vérifier que train et test ont exactement les mêmes colonnes

Résultat attendu : `Colonnes identiques : True`

In [9]:
print("========== COMPATIBILITÉ TRAIN / TEST ==========")
print("Colonnes identiques :", list(X_train.columns) == list(X_test.columns))

========== COMPATIBILITÉ TRAIN / TEST ==========
Colonnes identiques : True


### 4.6 — Importer les modèles

6 modèles de régression comparés.

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

print("Modèles importés.")

Modèles importés.


### 4.7 — Définir les modèles de référence

Paramètres volontairement standards, **pas encore optimisés**.

In [11]:
models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM": LGBMRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=-1,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),

    "CatBoost": CatBoostRegressor(
        iterations=200,
        learning_rate=0.05,
        depth=6,
        random_seed=42,
        verbose=False
    )
}

print("Nombre de modèles :", len(models))

Nombre de modèles : 6


### 4.8 — Entraîner les modèles

On mesure le temps d'entraînement.

In [12]:
trained_models = {}
training_times = {}

for name, model in models.items():

    print(f"\nEntraînement : {name}")

    start = time.time()
    model.fit(X_train, y_train)
    end = time.time()

    trained_models[name] = model
    training_times[name] = end - start

    print(f"Terminé en {training_times[name]:.2f} secondes")


Entraînement : Linear Regression


Terminé en 0.44 secondes

Entraînement : Random Forest


Terminé en 260.94 secondes

Entraînement : Gradient Boosting


Terminé en 247.95 secondes

Entraînement : XGBoost


Terminé en 3.57 secondes

Entraînement : LightGBM


Terminé en 2.77 secondes

Entraînement : CatBoost


Terminé en 6.35 secondes


### 4.9 — Faire les prédictions

In [13]:
predictions = {}
prediction_times = {}

for name, model in trained_models.items():

    start = time.time()
    predictions[name] = model.predict(X_test)
    end = time.time()

    prediction_times[name] = end - start

### 4.10 — Calculer les métriques

Problème : régression (prédiction de `fillRate_target_t_plus_1`, valeur numérique).

Métriques : **MAE**, **RMSE**, **R²**, **MAPE** (interprété avec prudence lorsque les valeurs réelles sont proches de zéro).

In [14]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

results = []

for name in trained_models:

    y_pred = predictions[name]

    mae = mean_absolute_error(y_test, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_test, y_pred)
    )

    r2 = r2_score(y_test, y_pred)

    # MAPE sécurisé
    mask = y_test != 0
    mape = np.mean(
        np.abs(
            (y_test[mask] - y_pred[mask])
            / y_test[mask]
        )
    ) * 100

    results.append({
        "Modele": name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Temps_Entrainement": training_times[name],
        "Temps_Prediction": prediction_times[name]
    })

### 4.11 — Afficher le classement

⚠️ On ne cherche pas encore 90 %. On veut une **référence honnête** : un résultat propre et généralisable vaut mieux qu'un 99 % obtenu par fuite de données.

In [15]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="R2",
    ascending=False
).reset_index(drop=True)

print(results_df)

              Modele        MAE       RMSE        MAPE        R2  Temps_Entrainement  Temps_Prediction
0           CatBoost  11.603956  15.456159  116.082733  0.297330            6.351866          0.021709
1            XGBoost  11.493667  15.475427  115.240715  0.295577            3.567642          0.087690
2           LightGBM  11.534908  15.479357  115.491954  0.295219            2.766029          0.127613
3  Gradient Boosting  11.654998  15.484307  116.330150  0.294769          247.945277          0.203012
4      Random Forest  11.684497  15.747485  115.476863  0.270592          260.939995          1.831758
5  Linear Regression  11.981364  15.770569  118.720155  0.268452            0.442399          0.051482


### 4.12 — Sauvegarder les résultats de référence

In [16]:
os.makedirs("../results", exist_ok=True)

results_df.to_excel(
    "../results/comparaison_modeles_reference.xlsx",
    index=False
)

print("Résultats de référence sauvegardés.")

Résultats de référence sauvegardés.


## Étape 5 — Optimisation des hyperparamètres

### 5.0 — Choix du modèle à optimiser

On n'optimise pas les 6 modèles au hasard. On repart du classement de l'étape 4 et on sélectionne les 2-3 meilleurs selon R², MAE et RMSE.

**Classement réel obtenu (étape 4, sans fuite) :**

| Rang | Modèle | R² | MAE | RMSE |
|---|---|---|---|---|
| 1 | CatBoost | 0.2973 | 11.60 | 15.46 |
| 2 | XGBoost | 0.2956 | 11.49 | 15.48 |
| 3 | LightGBM | 0.2952 | 11.53 | 15.48 |
| 4 | Gradient Boosting | 0.2948 | 11.65 | 15.48 |
| 5 | Random Forest | 0.2706 | 11.68 | 15.75 |
| 6 | Linear Regression | 0.2685 | 11.98 | 15.77 |

Les 3 premiers sont très proches. **Décision :** on optimise **LightGBM** et **XGBoost**, les deux boosters les plus rapides du groupe de tête (compromis performance/vitesse).

### 5.1 — Validation temporelle (TimeSeriesSplit)

On respecte l'ordre du temps : on ne mélange jamais aléatoirement, car l'objectif est de prédire le remplissage futur.

In [17]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

print(tscv)

TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None)


### 5.2 — Optimiser LightGBM

#### Espace de recherche (paramètres importants de LightGBM)

In [18]:
from sklearn.model_selection import RandomizedSearchCV
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

param_grid_lgbm = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "num_leaves": [31, 63, 127],
    "max_depth": [-1, 5, 10, 15],
    "min_child_samples": [10, 20, 40],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

#### Lancer la recherche (RandomizedSearchCV + validation temporelle)

In [19]:
random_search_lgbm = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_grid_lgbm,
    n_iter=20,
    scoring="r2",
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

In [20]:
import time as _t
debut = _t.time()

random_search_lgbm.fit(X_train, y_train)

print(f"Recherche terminée en {(_t.time() - debut) / 60:.2f} minutes")

Fitting 5 folds for each of 20 candidates, totalling 100 fits


Recherche terminée en 7.87 minutes


#### Voir les meilleurs paramètres

In [21]:
print("Meilleurs paramètres :")
print(random_search_lgbm.best_params_)

print("\nMeilleur score CV :")
print(random_search_lgbm.best_score_)

Meilleurs paramètres :
{'subsample': 0.9, 'num_leaves': 31, 'n_estimators': 200, 'min_child_samples': 20, 'max_depth': 5, 'learning_rate': 0.03, 'colsample_bytree': 0.9}

Meilleur score CV :
0.29277500308879556


### 5.3 — Évaluer sur notre vrai TEST

⚠️ Le `best_score_` de la validation n'est pas notre résultat final. On récupère le meilleur modèle et on l'évalue sur le jeu TEST jamais vu.

In [22]:
best_lgbm = random_search_lgbm.best_estimator_

y_pred_lgbm = best_lgbm.predict(X_test)

In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_lgbm = mean_absolute_error(y_test, y_pred_lgbm)

rmse_lgbm = np.sqrt(
    mean_squared_error(y_test, y_pred_lgbm)
)

r2_lgbm = r2_score(
    y_test,
    y_pred_lgbm
)

mask = y_test != 0

mape_lgbm = np.mean(
    np.abs(
        (y_test[mask] - y_pred_lgbm[mask])
        / y_test[mask]
    )
) * 100

print("========== LIGHTGBM OPTIMISÉ ==========")
print(f"MAE  : {mae_lgbm:.4f}")
print(f"RMSE : {rmse_lgbm:.4f}")
print(f"MAPE : {mape_lgbm:.2f}%")
print(f"R²   : {r2_lgbm:.4f}")

========== LIGHTGBM OPTIMISÉ ==========
MAE  : 11.5791
RMSE : 15.4425
MAPE : 115.77%
R²   : 0.2986


#### Comparaison baseline vs optimisé

L'objectif est d'obtenir une **vraie amélioration généralisable**, pas de forcer un score artificiel.

In [24]:
ligne_base = results_df[results_df["Modele"] == "LightGBM"].iloc[0]

comparaison_lgbm = pd.DataFrame({
    "Phase": ["Baseline", "Optimisé"],
    "MAE": [ligne_base["MAE"], mae_lgbm],
    "RMSE": [ligne_base["RMSE"], rmse_lgbm],
    "MAPE": [ligne_base["MAPE"], mape_lgbm],
    "R2": [ligne_base["R2"], r2_lgbm],
})

comparaison_lgbm

,Phase,MAE,RMSE,MAPE,R2
0,Baseline,11.534908,15.479357,115.491954,0.295219
1,Optimisé,11.579088,15.442533,115.768809,0.298569


### 5.4 — Optimiser XGBoost

Même démarche que pour LightGBM : XGBoost baseline → RandomizedSearchCV → TimeSeriesSplit → meilleurs hyperparamètres → évaluation sur TEST → comparaison avec LightGBM.

**Modèle à battre (LightGBM optimisé) :**

| Métrique | Valeur |
|---|---|
| MAE | 11.5791 |
| RMSE | 15.4425 |
| MAPE | 115.77 % |
| R² TEST | 0.2986 |

In [25]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

print("XGBoost importé.")

XGBoost importé.


In [26]:
# Validation temporelle (déjà définie, on la réutilise)
print(tscv)

TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None)


In [27]:
xgb = XGBRegressor(
    random_state=42,
    n_jobs=-1
)

print(xgb)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=-1, num_parallel_tree=None, ...)


In [28]:
param_grid_xgb = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 4, 6, 8],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

print("Espace de recherche défini.")

Espace de recherche défini.


In [29]:
random_search_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid_xgb,
    n_iter=20,
    scoring="r2",
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

In [30]:
import time as _t2
debut = _t2.time()

random_search_xgb.fit(X_train, y_train)

print(f"Recherche terminée en {(_t2.time() - debut) / 60:.2f} minutes")

Fitting 5 folds for each of 20 candidates, totalling 100 fits


Recherche terminée en 3.13 minutes


In [31]:
print("Meilleurs paramètres :")
print(random_search_xgb.best_params_)

print("\nMeilleur score CV :")
print(random_search_xgb.best_score_)

Meilleurs paramètres :
{'subsample': 0.7, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.01, 'colsample_bytree': 0.8}

Meilleur score CV :
0.2925065372485161


In [32]:
best_xgb = random_search_xgb.best_estimator_

print("Meilleur XGBoost récupéré.")

Meilleur XGBoost récupéré.


In [33]:
y_pred_xgb = best_xgb.predict(X_test)

In [34]:
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

rmse_xgb = np.sqrt(
    mean_squared_error(y_test, y_pred_xgb)
)

r2_xgb = r2_score(
    y_test,
    y_pred_xgb
)

mask = y_test != 0

mape_xgb = np.mean(
    np.abs(
        (y_test[mask] - y_pred_xgb[mask])
        / y_test[mask]
    )
) * 100

In [35]:
print("========== XGBOOST OPTIMISÉ ==========")
print(f"MAE  : {mae_xgb:.4f}")
print(f"RMSE : {rmse_xgb:.4f}")
print(f"MAPE : {mape_xgb:.2f}%")
print(f"R²   : {r2_xgb:.4f}")

========== XGBOOST OPTIMISÉ ==========
MAE  : 11.6695
RMSE : 15.4670
MAPE : 116.37%
R²   : 0.2963


In [36]:
ligne_base_xgb = results_df[results_df["Modele"] == "XGBoost"].iloc[0]

In [37]:
comparaison_xgb = pd.DataFrame({
    "Phase": ["LightGBM optimisé", "XGBoost baseline", "XGBoost optimisé"],
    "MAE": [mae_lgbm, ligne_base_xgb["MAE"], mae_xgb],
    "RMSE": [rmse_lgbm, ligne_base_xgb["RMSE"], rmse_xgb],
    "MAPE": [mape_lgbm, ligne_base_xgb["MAPE"], mape_xgb],
    "R2": [r2_lgbm, ligne_base_xgb["R2"], r2_xgb],
})

comparaison_xgb

,Phase,MAE,RMSE,MAPE,R2
0,LightGBM optimisé,11.579088,15.442533,115.768809,0.298569
1,XGBoost baseline,11.493667,15.475427,115.240715,0.295577
2,XGBoost optimisé,11.669521,15.466982,116.370388,0.296346


### 5.5 — Conclusion : meilleur modèle optimisé

On récapitule les performances baseline vs optimisé sur le TEST jamais vu, **sans la fuite de données** (statut_prevu et priorite_recommandee exclues).

**Résultats obtenus :**

| Modèle | Phase | MAE | RMSE | MAPE | R² TEST |
|---|---|---|---|---|---|
| LightGBM | Baseline | 11.5349 | 15.4794 | 115.49 % | 0.2952 |
| LightGBM | Optimisé | 11.5791 | 15.4425 | 115.77 % | 0.2986 |
| XGBoost | Baseline | 11.4937 | 15.4754 | 115.24 % | 0.2956 |
| XGBoost | Optimisé | 11.6695 | 15.4670 | 116.37 % | 0.2963 |

Après correction de la fuite, l'optimisation n'apporte qu'un **gain marginal** :
RMSE et R² s'améliorent légèrement pour LightGBM (R² 0.2952 → 0.2986), MAE/MAPE
restent quasi stables. Le R² reste proche de **0.30** : c'est le résultat honnête,
reproductible et généralisable — le R² très élevé (0.83) observé avant correction
provenait uniquement des colonnes fuyantes, pas d'un pouvoir prédictif réel.

> ⚠️ Le MAPE est très élevé (~116 %) car certaines valeurs réelles sont proches de
> zéro ; à interpréter avec prudence. On privilégie MAE et RMSE.

In [38]:
synthese_finale = pd.DataFrame({
    "Modele": [
        "LightGBM baseline",
        "LightGBM optimisé",
        "XGBoost baseline",
        "XGBoost optimisé",
    ],
    "MAE": [ligne_base["MAE"], mae_lgbm, ligne_base_xgb["MAE"], mae_xgb],
    "RMSE": [ligne_base["RMSE"], rmse_lgbm, ligne_base_xgb["RMSE"], rmse_xgb],
    "MAPE": [ligne_base["MAPE"], mape_lgbm, ligne_base_xgb["MAPE"], mape_xgb],
    "R2": [ligne_base["R2"], r2_lgbm, ligne_base_xgb["R2"], r2_xgb],
})

synthese_finale = synthese_finale.sort_values("R2", ascending=False)
synthese_finale = synthese_finale.reset_index(drop=True)

synthese_finale

,Modele,MAE,RMSE,MAPE,R2
0,LightGBM optimisé,11.579088,15.442533,115.768809,0.298569
1,XGBoost optimisé,11.669521,15.466982,116.370388,0.296346
2,XGBoost baseline,11.493667,15.475427,115.240715,0.295577
3,LightGBM baseline,11.534908,15.479357,115.491954,0.295219


In [41]:
gagnant = synthese_finale.iloc[0]

print("=" * 60)
print("MEILLEUR MODÈLE OPTIMISÉ :", gagnant["Modele"])
print(
    f"MAE : {gagnant['MAE']:.4f} | RMSE : {gagnant['RMSE']:.4f} | "
    f"MAPE : {gagnant['MAPE']:.2f} % | R² : {gagnant['R2']:.4f}"
)
print("=" * 60)

nom_baseline = gagnant["Modele"].replace(" optimisé", " baseline")
baseline_gagnant = synthese_finale[synthese_finale["Modele"] == nom_baseline].iloc[0]
print(
    f"Gain vs {nom_baseline} : "
    f"R² {gagnant['R2'] - baseline_gagnant['R2']:+.5f}, "
    f"MAE {gagnant['MAE'] - baseline_gagnant['MAE']:+.4f}"
)

MEILLEUR MODÈLE OPTIMISÉ : LightGBM optimisé
MAE : 11.5791 | RMSE : 15.4425 | MAPE : 115.77 % | R² : 0.2986
Gain vs LightGBM baseline : R² +0.00335, MAE +0.0442
